In [ ]:
# 加载环境变量
from dotenv import load_dotenv

load_dotenv()

一个完整的Agent至少要包含两个关键的部分：
- **模型**：是Agent的大脑，负责推理、分析，规划任务步骤
- **工具**：是Agent的手脚，负责执行任务，与外界交互

因此，定义带有工具的Agent的基本流程如下：
- 定义工具
- 初始化模型
- 初始化Agent，绑定模型和工具

如果想要最简化的了解tool的作用方法，查看2.1中间相关的例子就可以明白了

# 1.自定义工具

所谓的**工具（Tool）**，本质就是一个可调用的**函数**，但是这个函数不是我们自己去调用，而是给模型调用。因此除了定义函数外，我们还需要清晰描述这个工具，让模型知道这个工具如何使用。包括下列信息：
- 工具名
- 工具的作用
- 工具需要的参数

在Langchain中，我们是把工具都是在本地内部的，模型信息是在云端的。所以在第一次的信息交流中，是先将拥有的工具以及工具的信息发送给大模型，让大模型在需要调用的时候调用对应的模型。

## 1.1.基于tool描述工具

在LangChain中，定义工具需要用到@tool装饰器，我们可以通过装饰器来定义工具名、工具的作用：


In [ ]:
from langchain_core.tools import tool

# 工具和工具的名字
@tool("square_root", description="Calculate the square root of a number")
def tool1(x: float) -> float:
    return x ** 0.5

## 1.2.使用函数名和文档注释描述工具

**这个才是我们更加常用的，基于函数名来确定tool**

如果不@tool装饰器没有定义工具名和作用描述，此时：
- 工具名：默认就是函数名
- 工具所需的参数：默认就是函数的参数列表
- 工具作用的描述：默认就是函数的文档注释

In [ ]:
from langchain_core.tools import tool
# 通过tool装饰器定义工具
@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

In [ ]:
# 定义一个查询天气的tool
@tool
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """
    需要描述的是工具的名字以及工具的参数
    Get current weather and optional forecast.
    Args:
        location: city name or coordinates
        units: unit of degrees
        include_forecast: does it include the weather forecast
    """
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result

# 1.3.定义Pydantic Model描述参数
如果函数的参数比较多，而且比较复杂，此时建议通过pydantic model来描述参数列表。

在遇到比较复杂的参数/对于相关参数的类型需要限制的时候使用。


In [ ]:
# 通过自定义model来约束入参
from pydantic import BaseModel, Field
from typing import Literal


# 例如一个查询天气的tool
class WeatherInput(BaseModel):
    """查询天气的输入参数."""
    location: str = Field(description="City name or coordinates")
    units: Literal["celsius", "fahrenheit"] = Field(# Literal是类型提示的含义，约束只能是celsius或者fahrenheit
        default="celsius",# 约束只能是celsius或者fahrenheit
        description="Temperature unit preference, default is celsius."
    )
    include_forecast: bool = Field(
        default=False,
        description="Include 5-day forecast"
    )

# 定义一个查询天气的tool
@tool(args_schema=WeatherInput)#指出参数的限制就是上面我们设置的WeatherInput
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """Get current weather and optional forecast."""
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result


工具调用方式与普通函数调用方式一致。


In [ ]:
square_root.invoke({"x": 467})

In [ ]:
get_weather.invoke({"location": "杭州", "include_forecast": True})

## 测试

In [ ]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(
    model="deepseek-chat",
    tools=[square_root, get_weather],
    system_prompt="你可以使用工具回答用户问题，调用工具时尽量使用默认参数，除非用户特别指定。"
)

In [ ]:
for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="杭州接下来几天天气如何?")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)


In [ ]:
response = agent.invoke(
    {"messages": [HumanMessage(content="467和529的平方根是多少?")]},
)

for message in response['messages']:
    print(message.pretty_print())

观察上面内容的输出结果，我们可以看到：模型其实没有方法在云端自主调用tool，使用的方法是要把需要本地访问的工具以及申请的相关参数发送到哦本地在本地进行运算。


完整流程如图：
<img src="./resources/agent-flow2.png">

# 2.预定义Tool

LangChain中提供了很多预定义的Tool，方便我们使用。例如：
- tavily：就是一个用来做web搜索的工具

## 2.1.基本用法
它的使用步骤是这样的：
- 注册账号，创建API_KEY
- 配置环境变量: TAVILY_API_KEY
- 安装依赖：`uv add langchain-tavily`


In [ ]:
# 使用tavily作为web搜索工具
from langchain_tavily import TavilySearch

search_tool = TavilySearch(#直接得到了这个工具
    max_results=5,
    topic="general", # general, news, finance
    # include_answer=False,
    # include_raw_content=False,
    # include_images=False,
    # include_image_descriptions=False,
    # search_depth="basic",
    # time_range="day",
    # include_domains=None,
    # exclude_domains=None
)

In [ ]:
search_tool.invoke("ootd是什么梗？")

In [ ]:
# 创建智能体，使用预定义工具tavily
agent = create_agent(
    model="deepseek-chat",
    tools=[search_tool],
    system_prompt="你是一个智能助手，你使用工具来解决用户问题。"
)

In [ ]:
response = agent.invoke(
    {"messages": [HumanMessage(content="ootd是什么梗？")]},
)

for message in response['messages']:
    message.pretty_print()

## 2.2.优化

目前的搜索智能体存在两个问题：
- 官方默认的tavily工具过于复杂（返回的数据很多，但是我们真正使用的数据不多）
- 结果中不包含网页数据源，可信度低

解决思路：
- 自定义tavily工具（自定义来）
- 结构化输出

### 自定义tavily工具

LangChain官方提供的tavily工具包含了完整的参数列表，会导致额外的流量和Token消耗。因此，对于简单的业务，我们建议大家利用tavily自定义工具。


In [ ]:
# 先使用官方的客户端做初始化
tavily = TavilySearch(
    # 我们前面调用官方的数据的时候相当于直接使用了Tavily所有的数据
    # 现在我们写死了Tavily使用到的数据，于是能够减少我们在实际使用的时候token的消耗
    max_results=5,
    topic="general"
)

# 然后自己封装为tool
@tool
def web_search(query: str):#这个就是我们前面的最原始的tool定义方法，使用前面tavily得到的结果进行分析
    """Search the web for information"""
    return tavily.invoke(query)# 还是调用了tavily的数据
# 使用原始定义的方法来创建工具

### 定义结构化输出实体


In [ ]:
# 我们上一章提到的使用pydantic来对于输出进行约束
from pydantic import BaseModel, Field

# Agent回答内容引用的网页信息
class Reference(BaseModel):
    title: str = Field(description="The title of the web page cited in the answer")
    url: str = Field(description="The url of the web page cited in the answer")

# Agent的回答内容
class AnswerInfo (BaseModel):
    answer: str = Field(description="The final answer for user")
    reference: list[Reference] = Field(description="The web pages cited in the answer")

In [ ]:
# 创建智能体，使用预定义工具tavily
agent = create_agent(
    model="deepseek-chat",
    tools=[web_search],
    system_prompt="你是一个智能助手，你使用工具来解决用户问题。",
    response_format=AnswerInfo# 返回的格式
)

In [ ]:
# 调用agent
response = agent.invoke(
    {"messages": [HumanMessage(content="蒸蚌是什么梗？")]},
)

# 获取结构化输出
print(response['structured_response'])

在你研究完成上面的大模型的调用的时候想必你会出现一个问题：Pydantic Model的作用究竟是怎样的
- 在我们使用tool的时候，使用来为大模型使用的参数进行限制的，即告诉大模型“这个函数需要什么名字的参数、这些参数是什么数据类型，以及这些参数代表什么含义”的详细需求说明书。
- 同时我们也将其使用于agent的定义中。
```python 
class AnswerInfo (BaseModel):
    answer: str = Field(description="The final answer for user")
    # 🌟 关键应用点：Reference 作为 list 的元素类型嵌套在这里
    reference: list[Reference] = Field(description="The web pages cited in the answer")

agent = create_agent(
    model="deepseek-chat",
    tools=[web_search],
    system_prompt="你是一个智能助手，你使用工具来解决用户问题。",
    response_format=AnswerInfo # 👈 整个格式规格被注入智能体
)
```

其实归根结底，LangChain 自动把 Pydantic Model 翻译成 JSON Schema（一种标准的数据描述文件），然后发给大模型作为“需求说明书”。

然后我们在调用tool的时候以及设置agent输出要求的时候，我们都会调用这个 JSON Schema 来对于我们输入输出的结果给出限制，从而实现对于相关部分的限制。